# 54. 持续预训练 Replay：怎样量化新知识学习与旧知识遗忘？

## 面试回答主线

持续预训练不能只看新语料 loss，因为模型可能在学习新域的同时忘掉旧域能力。一个可操作的评测要保存训练前 checkpoint，在固定旧知识集与新知识集上分别测 next-token accuracy，并用相同起点比较 new-only 与 replay。Replay 将具有代表性的旧样本混入新语料，使梯度同时约束旧决策边界；buffer 若只覆盖一种旧答案，未覆盖能力仍会遗忘。面试中我会实现一个小型 PyTorch next-token 模型，真实执行旧域预训练、新域续训、平衡 replay 与偏置 replay，并输出每个样本的预测变化。Replay 比例需要在可塑性与稳定性之间调节，不是越大越好。生产系统还要处理 tokenizer 漂移、数据去重、采样权重、优化器状态与大规模回归评测。

## 1. 真实案例：旧产品知识与新治理知识的下一 token 预测

旧域包含三条“保修→两年”和三条“配送→次日”；新域包含三条“隐私数据→需脱敏”和三条“高风险动作→需审批”。三维特征前两维表示语义簇，第三维表示新治理域；输出是四个可读答案 token。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示旧域与新域语料
import copy  # 导入深复制工具从同一预训练 checkpoint 创建公平分支
import torch  # 导入 PyTorch 实现真实 next-token forward 与持续训练
from torch import nn  # 导入神经网络模块构造微型语言模型
from torch.nn import functional as F  # 导入交叉熵损失训练下一 token 分类
torch.set_num_threads(1)  # 限制教学训练线程数以获得稳定执行时间
torch.manual_seed(7)  # 固定模型初始化与遗忘轨迹
answer_tokens = ["两年", "次日", "需脱敏", "需审批"]  # 定义旧域与新域的四个真实答案 token
old_cases = [{"id": "O01", "prompt": "产品 A 保修多久", "features": [-2.0, -1.0, 0.0], "answer": 0}, {"id": "O02", "prompt": "电池保修期多久", "features": [-1.5, -0.8, 0.0], "answer": 0}, {"id": "O03", "prompt": "延保方案默认时长", "features": [-1.0, -1.5, 0.0], "answer": 0}, {"id": "O04", "prompt": "标准件何时送达", "features": [2.0, 1.0, 0.0], "answer": 1}, {"id": "O05", "prompt": "加急件配送承诺", "features": [1.5, 0.8, 0.0], "answer": 1}, {"id": "O06", "prompt": "同城订单预计时效", "features": [1.0, 1.5, 0.0], "answer": 1}]  # 定义六条需要长期保留的旧产品知识
new_cases = [{"id": "N01", "prompt": "日志入湖前如何处理", "features": [-2.0, -1.0, 1.0], "answer": 2}, {"id": "N02", "prompt": "训练语料含手机号", "features": [-1.5, -0.8, 1.0], "answer": 2}, {"id": "N03", "prompt": "客诉文本进入分析库", "features": [-1.0, -1.5, 1.0], "answer": 2}, {"id": "N04", "prompt": "高风险工具调用前", "features": [2.0, 1.0, 1.0], "answer": 3}, {"id": "N05", "prompt": "删除知识库之前", "features": [1.5, 0.8, 1.0], "answer": 3}, {"id": "N06", "prompt": "批量退款执行前", "features": [1.0, 1.5, 1.0], "answer": 3}]  # 定义六条需要持续学习的新治理知识
old_x = torch.tensor([item["features"] for item in old_cases], dtype=torch.float32)  # 构造旧域 prompt 特征矩阵
old_y = torch.tensor([item["answer"] for item in old_cases], dtype=torch.long)  # 构造旧域下一 token 标签
new_x = torch.tensor([item["features"] for item in new_cases], dtype=torch.float32)  # 构造新域 prompt 特征矩阵
new_y = torch.tensor([item["answer"] for item in new_cases], dtype=torch.long)  # 构造新域下一 token 标签
preview = [{"语料": "旧域", "样本": item["id"], "prompt": item["prompt"], "目标token": answer_tokens[item["answer"]]} for item in old_cases] + [{"语料": "新域", "样本": item["id"], "prompt": item["prompt"], "目标token": answer_tokens[item["answer"]]} for item in new_cases]  # 汇总十二条真实语义样本
print("持续预训练语料预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示旧知识与新知识的明确边界

持续预训练语料预览：
[{'语料': '旧域', '样本': 'O01', 'prompt': '产品 A 保修多久', '目标token': '两年'},
 {'语料': '旧域', '样本': 'O02', 'prompt': '电池保修期多久', '目标token': '两年'},
 {'语料': '旧域', '样本': 'O03', 'prompt': '延保方案默认时长', '目标token': '两年'},
 {'语料': '旧域', '样本': 'O04', 'prompt': '标准件何时送达', '目标token': '次日'},
 {'语料': '旧域', '样本': 'O05', 'prompt': '加急件配送承诺', '目标token': '次日'},
 {'语料': '旧域', '样本': 'O06', 'prompt': '同城订单预计时效', '目标token': '次日'},
 {'语料': '新域', '样本': 'N01', 'prompt': '日志入湖前如何处理', '目标token': '需脱敏'},
 {'语料': '新域', '样本': 'N02', 'prompt': '训练语料含手机号', '目标token': '需脱敏'},
 {'语料': '新域', '样本': 'N03', 'prompt': '客诉文本进入分析库', '目标token': '需脱敏'},
 {'语料': '新域', '样本': 'N04', 'prompt': '高风险工具调用前', '目标token': '需审批'},
 {'语料': '新域', '样本': 'N05', 'prompt': '删除知识库之前', '目标token': '需审批'},
 {'语料': '新域', '样本': 'N06', 'prompt': '批量退款执行前', '目标token': '需审批'}]


## 2. 第一阶段：真实预训练旧知识并保存 checkpoint

`TinyNextTokenLM` 用三维 prompt 表示经过小型 MLP 输出四个答案 token logits。先只训练旧域，使六条旧知识达到稳定正确，再深复制状态作为所有持续训练方案的同一起点。

In [2]:
class TinyNextTokenLM(nn.Module):  # 定义具有共享隐藏层的微型下一 token 模型
    def __init__(self):  # 初始化三维输入、四维瓶颈与四 token 输出头
        super().__init__()  # 注册 PyTorch 模块参数
        self.network = nn.Sequential(nn.Linear(3, 4), nn.Tanh(), nn.Linear(4, len(answer_tokens)))  # 构造会在新旧域间共享参数的语言模型头
    def forward(self, prompt_features):  # 对一批 prompt 表示执行下一 token forward
        return self.network(prompt_features)  # 返回四个候选答案 token 的 logits
def accuracy(model, inputs, labels):  # 计算一个固定语料集的下一 token 准确率
    with torch.no_grad():  # 评估阶段不建立梯度图
        predictions = model(inputs).argmax(dim=-1)  # 选择当前模型概率最高的答案 token
    return float((predictions == labels).float().mean()), predictions.tolist()  # 返回准确率和逐样本预测 ID
def train_model(model, inputs, labels, steps=300):  # 用固定设置对指定语料执行真实梯度训练
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.05, weight_decay=0.05)  # 创建带权重衰减的持续预训练优化器
    history = []  # 记录训练损失和新旧准确率轨迹
    first_gradient = 0.0  # 初始化首步共享层梯度范数
    for step in range(steps):  # 执行有限步 next-token 训练
        logits = model(inputs)  # 对当前 replay 混合批次执行真实 forward
        loss = F.cross_entropy(logits, labels)  # 计算目标答案 token 的交叉熵
        optimizer.zero_grad()  # 清除上一轮累计梯度
        loss.backward()  # 让新旧语料共同更新共享模型参数
        if step == 0:  # 在第一次更新前记录梯度证据
            first_gradient = float(model.network[0].weight.grad.norm())  # 读取共享输入层的梯度范数
        optimizer.step()  # 应用持续预训练梯度
        if step in {0, 49, 99, 199, steps - 1}:  # 在关键训练阶段记录稳定性与可塑性
            old_acc, old_predictions = accuracy(model, old_x, old_y)  # 测量固定旧知识集的保留率
            new_acc, new_predictions = accuracy(model, new_x, new_y)  # 测量固定新知识集的学习率
            history.append({"step": step + 1, "loss": round(float(loss.detach()), 4), "old_acc": old_acc, "new_acc": new_acc})  # 保存可观察训练轨迹
    return history, first_gradient  # 返回损失准确率轨迹与首步梯度
pretrained_model = TinyNextTokenLM()  # 创建待学习旧产品知识的基础模型
pretrain_history, pretrain_gradient = train_model(pretrained_model, old_x, old_y)  # 对六条旧知识执行真实预训练
old_before_acc, old_before_predictions = accuracy(pretrained_model, old_x, old_y)  # 评估保存 checkpoint 前的旧知识能力
checkpoint = copy.deepcopy(pretrained_model.state_dict())  # 深复制预训练权重作为公平持续训练起点
print({"旧域预训练轨迹": pretrain_history, "首步共享层梯度": round(pretrain_gradient, 4), "checkpoint旧域准确率": old_before_acc, "逐样本预测": [answer_tokens[index] for index in old_before_predictions]})  # 展示预训练 forward、梯度与旧知识结果

{'旧域预训练轨迹': [{'step': 1, 'loss': 1.3141, 'old_acc': 0.5, 'new_acc': 0.5}, {'step': 50, 'loss': 0.0049, 'old_acc': 1.0, 'new_acc': 0.0}, {'step': 100, 'loss': 0.0044, 'old_acc': 1.0, 'new_acc': 0.0}, {'step': 200, 'loss': 0.0031, 'old_acc': 1.0, 'new_acc': 0.0}, {'step': 300, 'loss': 0.0023, 'old_acc': 1.0, 'new_acc': 0.0}], '首步共享层梯度': 0.344, 'checkpoint旧域准确率': 1.0, '逐样本预测': ['两年', '两年', '两年', '次日', '次日', '次日']}


## 3. Baseline（基线）：只训练新语料导致旧知识被覆盖

从保存的 checkpoint 出发，只对新治理语料训练。共享瓶颈和输出头逐渐把相似语义簇映射到新答案；新域达到 100%，但旧 prompt 也开始错误输出“需脱敏/需审批”。这就是只看新域 loss 会遗漏的灾难性遗忘。

In [3]:
new_only_model = TinyNextTokenLM()  # 创建 new-only 持续训练分支
new_only_model.load_state_dict(checkpoint)  # 从完全相同的旧域预训练 checkpoint 开始
new_only_history, new_only_gradient = train_model(new_only_model, new_x, new_y)  # 只用六条新治理知识继续训练
new_only_old_acc, new_only_old_predictions = accuracy(new_only_model, old_x, old_y)  # 测量新域训练后的旧知识保留率
new_only_new_acc, new_only_new_predictions = accuracy(new_only_model, new_x, new_y)  # 测量新域知识学习率
forgetting = old_before_acc - new_only_old_acc  # 计算从预训练 checkpoint 到持续训练后的绝对遗忘量
print({"new-only训练轨迹": new_only_history, "首步共享层梯度": round(new_only_gradient, 4), "新域准确率": new_only_new_acc, "旧域准确率": new_only_old_acc, "遗忘量": forgetting, "旧问题现在输出": [answer_tokens[index] for index in new_only_old_predictions]})  # 展示新知识成功与旧知识丢失同时发生

{'new-only训练轨迹': [{'step': 1, 'loss': 7.03, 'old_acc': 1.0, 'new_acc': 0.0}, {'step': 50, 'loss': 0.0168, 'old_acc': 0.0, 'new_acc': 1.0}, {'step': 100, 'loss': 0.014, 'old_acc': 0.0, 'new_acc': 1.0}, {'step': 200, 'loss': 0.0114, 'old_acc': 0.0, 'new_acc': 1.0}, {'step': 300, 'loss': 0.0092, 'old_acc': 0.0, 'new_acc': 1.0}], '首步共享层梯度': 0.5791, '新域准确率': 1.0, '旧域准确率': 0.0, '遗忘量': 1.0, '旧问题现在输出': ['需脱敏', '需脱敏', '需脱敏', '需审批', '需审批', '需审批']}


## 4. 手写 Replay：在同一 batch 中混合平衡旧知识

Replay 分支仍从同一 checkpoint 开始，把六条新样本与覆盖两类旧答案的六条旧样本拼成训练批次。核心不是简单复制数据，而是确保旧能力类别在梯度中持续出现。

In [4]:
replay_model = TinyNextTokenLM()  # 创建平衡 replay 持续训练分支
replay_model.load_state_dict(checkpoint)  # 从同一旧域 checkpoint 开始保证公平比较
replay_x = torch.cat([new_x, old_x], dim=0)  # 将全部新语料与代表性旧样本混合成 replay 批次
replay_y = torch.cat([new_y, old_y], dim=0)  # 对齐新旧样本的下一 token 标签
replay_history, replay_gradient = train_model(replay_model, replay_x, replay_y)  # 用新旧混合梯度执行持续预训练
replay_old_acc, replay_old_predictions = accuracy(replay_model, old_x, old_y)  # 评估 replay 后旧产品知识保留率
replay_new_acc, replay_new_predictions = accuracy(replay_model, new_x, new_y)  # 评估 replay 后新治理知识学习率
replay_forgetting = old_before_acc - replay_old_acc  # 计算平衡 replay 的绝对遗忘量
print({"平衡replay训练轨迹": replay_history, "首步共享层梯度": round(replay_gradient, 4), "新域准确率": replay_new_acc, "旧域准确率": replay_old_acc, "遗忘量": replay_forgetting, "旧域预测": [answer_tokens[index] for index in replay_old_predictions], "新域预测": [answer_tokens[index] for index in replay_new_predictions]})  # 展示稳定性与可塑性同时保留

{'平衡replay训练轨迹': [{'step': 1, 'loss': 3.5161, 'old_acc': 1.0, 'new_acc': 0.0}, {'step': 50, 'loss': 0.4297, 'old_acc': 0.6666666865348816, 'new_acc': 1.0}, {'step': 100, 'loss': 0.0476, 'old_acc': 1.0, 'new_acc': 1.0}, {'step': 200, 'loss': 0.0205, 'old_acc': 1.0, 'new_acc': 1.0}, {'step': 300, 'loss': 0.0158, 'old_acc': 1.0, 'new_acc': 1.0}], '首步共享层梯度': 0.2892, '新域准确率': 1.0, '旧域准确率': 1.0, '遗忘量': 0.0, '旧域预测': ['两年', '两年', '两年', '次日', '次日', '次日'], '新域预测': ['需脱敏', '需脱敏', '需脱敏', '需审批', '需审批', '需审批']}


## 5. 结果解读：逐样本比较 checkpoint、new-only 与 replay

下面逐条列出旧知识在三阶段的预测，再列出新知识在两种持续训练方案中的预测。new-only 并非训练失败：它的新域结果完全正确；问题是缺少旧域约束。Replay 的价值必须通过双域评测才能看见。

In [5]:
result_rows = []  # 收集旧域六条知识的逐样本遗忘对照
for index, item in enumerate(old_cases):  # 遍历固定旧知识评测集
    result_rows.append({"样本": item["id"], "prompt": item["prompt"], "目标": answer_tokens[item["answer"]], "预训练checkpoint": answer_tokens[old_before_predictions[index]], "new-only后": answer_tokens[new_only_old_predictions[index]], "replay后": answer_tokens[replay_old_predictions[index]]})  # 保存每条旧知识的预测迁移
new_result_rows = []  # 收集新域六条知识的学习结果
for index, item in enumerate(new_cases):  # 遍历固定新知识评测集
    new_result_rows.append({"样本": item["id"], "prompt": item["prompt"], "目标": answer_tokens[item["answer"]], "new-only": answer_tokens[new_only_new_predictions[index]], "replay": answer_tokens[replay_new_predictions[index]]})  # 保存两种方案的新知识预测
print("旧知识逐样本遗忘对照：")  # 输出结果解读的旧域标题
pprint(result_rows, sort_dicts=False)  # 展示 new-only 如何覆盖旧答案而 replay 保留
print("新知识逐样本学习结果：")  # 输出结果解读的新域标题
pprint(new_result_rows, sort_dicts=False)  # 确认 replay 没有阻止新域学习

旧知识逐样本遗忘对照：
[{'样本': 'O01',
  'prompt': '产品 A 保修多久',
  '目标': '两年',
  '预训练checkpoint': '两年',
  'new-only后': '需脱敏',
  'replay后': '两年'},
 {'样本': 'O02',
  'prompt': '电池保修期多久',
  '目标': '两年',
  '预训练checkpoint': '两年',
  'new-only后': '需脱敏',
  'replay后': '两年'},
 {'样本': 'O03',
  'prompt': '延保方案默认时长',
  '目标': '两年',
  '预训练checkpoint': '两年',
  'new-only后': '需脱敏',
  'replay后': '两年'},
 {'样本': 'O04',
  'prompt': '标准件何时送达',
  '目标': '次日',
  '预训练checkpoint': '次日',
  'new-only后': '需审批',
  'replay后': '次日'},
 {'样本': 'O05',
  'prompt': '加急件配送承诺',
  '目标': '次日',
  '预训练checkpoint': '次日',
  'new-only后': '需审批',
  'replay后': '次日'},
 {'样本': 'O06',
  'prompt': '同城订单预计时效',
  '目标': '次日',
  '预训练checkpoint': '次日',
  'new-only后': '需审批',
  'replay后': '次日'}]
新知识逐样本学习结果：
[{'样本': 'N01',
  'prompt': '日志入湖前如何处理',
  '目标': '需脱敏',
  'new-only': '需脱敏',
  'replay': '需脱敏'},
 {'样本': 'N02',
  'prompt': '训练语料含手机号',
  '目标': '需脱敏',
  'new-only': '需脱敏',
  'replay': '需脱敏'},
 {'样本': 'N03',
  'prompt': '客诉文本进入分析库',
  '目标': '需脱敏',
  'new-only'

## 6. 失败案例与修正：Replay buffer 只覆盖一种旧答案

若 buffer 只抽到前三条“保修→两年”，模型会保留这类旧知识，却忘掉未覆盖的“配送→次日”。总 replay 样本数看似不少，类别覆盖仍然失衡。修正是按能力簇、来源和时间分层采样，并在固定旧评测集上按 slice 报告遗忘。

In [6]:
biased_model = TinyNextTokenLM()  # 创建偏置 replay 的失败分支
biased_model.load_state_dict(checkpoint)  # 从同一预训练 checkpoint 开始复现实验
biased_x = torch.cat([new_x, old_x[:3]], dim=0)  # 只回放旧域中“保修→两年”这一能力簇
biased_y = torch.cat([new_y, old_y[:3]], dim=0)  # 对齐偏置 buffer 的下一 token 标签
biased_history, biased_gradient = train_model(biased_model, biased_x, biased_y)  # 用覆盖不完整的 replay buffer 持续训练
biased_old_acc, biased_old_predictions = accuracy(biased_model, old_x, old_y)  # 评估全部旧知识而不是只测被回放样本
warranty_acc = sum(prediction == label for prediction, label in zip(biased_old_predictions[:3], old_y[:3].tolist())) / 3  # 计算被 buffer 覆盖的保修知识准确率
delivery_acc = sum(prediction == label for prediction, label in zip(biased_old_predictions[3:], old_y[3:].tolist())) / 3  # 计算未被 buffer 覆盖的配送知识准确率
buffer_coverage = {answer_tokens[label]: int((biased_y == label).sum()) for label in sorted(set(old_y.tolist()))}  # 统计旧能力类别在 replay 中的覆盖数量
print({"失败_buffer旧类覆盖": buffer_coverage, "总体旧域准确率": biased_old_acc, "保修slice准确率": warranty_acc, "配送slice准确率": delivery_acc, "失败后旧域预测": [answer_tokens[index] for index in biased_old_predictions], "修正": "按答案能力簇分层回放六条旧样本"})  # 展示平均指标掩盖的 slice 遗忘与修正

{'失败_buffer旧类覆盖': {'两年': 3, '次日': 0}, '总体旧域准确率': 0.5, '保修slice准确率': 1.0, '配送slice准确率': 0.0, '失败后旧域预测': ['两年', '两年', '两年', '需审批', '需审批', '需审批'], '修正': '按答案能力簇分层回放六条旧样本'}


## 7. 生产差距与最小回归检查

真实持续预训练要在 token 级语言模型 loss 上混合海量语料，并管理 tokenizer 扩展、文档去重、污染、时间衰减和数据许可。Replay 比例过大会降低新域可塑性，过小则无法稳定旧能力；还可比较正则化、adapter 隔离和模型合并。优化器状态也会影响遗忘，公平实验必须从同一 checkpoint 与状态出发。最后的断言只验证本实验已展示的梯度、新域学习、new-only 遗忘、平衡 replay 与偏置 buffer。

In [7]:
assert len(old_cases) >= 5 and len(new_cases) >= 5  # 确认新旧语义样本数量都满足逐样本教学要求
assert pretrain_gradient > 0.0 and replay_gradient > 0.0  # 确认预训练与 replay 都通过真实反向传播更新共享参数
assert old_before_acc == 1.0  # 确认比较遗忘前旧知识已经被模型可靠学会
assert new_only_new_acc == 1.0 and new_only_old_acc < old_before_acc  # 确认 new-only 学到新知识同时真实遗忘旧知识
assert replay_new_acc == 1.0 and replay_old_acc == 1.0  # 确认平衡 replay 同时保留新旧两域能力
assert replay_forgetting < forgetting  # 确认 replay 相比 new-only 减少绝对遗忘量
assert warranty_acc == 1.0 and delivery_acc == 0.0  # 确认偏置 buffer 只保护覆盖能力而未覆盖 slice 仍遗忘
print("回归检查通过：持续训练梯度、双域准确率、平衡 replay 与偏置遗忘均已验证。")  # 输出最终验收结论

回归检查通过：持续训练梯度、双域准确率、平衡 replay 与偏置遗忘均已验证。
